In [0]:
# Install libraries
%pip install -U -qqqq databricks-openai databricks-agents


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart the runtime
dbutils.library.restartPython()

# LLM client

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()

openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Choose LLM endpoint

In [0]:
# from databricks.sdk import WorkspaceClient
# def is_endpoint_available(endpoint_name):
#   try:
#     client = WorkspaceClient().serving_endpoints.get_open_ai_client()
#     client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
#     return True
#   except Exception:
#     return False
  

# client = WorkspaceClient()
# for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
#     if is_endpoint_available(candidate_endpoint_name):
#       LLM_ENDPOINT_NAME = candidate_endpoint_name
# assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

In [0]:
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

# Get catalog and scheme information

In [0]:
# Zjistíme tvůj aktuální katalog a schéma v Unity Catalogu
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema = spark.sql("SELECT current_schema()").collect()[0][0]

UC_PREFIX = f"{current_catalog}.{current_schema}"
print(f"Tvůj agent bude číst tabulky z: {UC_PREFIX}")

Tvůj agent bude číst tabulky z: agent.default


# Functions

### Delta table fom UC

In [0]:
def max_revenue_product():
    df = spark.table(f"{UC_PREFIX}.delta_table")

    df = df.withColumn(
        "revenue",
        df.price * df.quantity
    )

    max_rev = (
        df.selectExpr("max(revenue) as max_revenue")
          .collect()[0]["max_revenue"]
    )

    result = (
        df.filter(df.revenue == max_rev)
          .select("product")
          .distinct()
          .collect()
    )

    return [row["product"] for row in result]




def average_price():
    df = spark.table(f"{UC_PREFIX}.delta_table")

    value = (
        df.selectExpr("avg(price) as avg_price")
          .collect()[0]["avg_price"]
    )

    return float(value)




def total_revenue():
    df = spark.table(f"{UC_PREFIX}.delta_table")

    df = df.withColumn(
        "revenue",
        df.price * df.quantity
    )

    value = (
        df.selectExpr("sum(revenue) as total_revenue")
          .collect()[0]["total_revenue"]
    )

    return float(value)




# Tool spec

In [0]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "max_revenue_product",
            "description": "Return product or products with the highest revenue (price multiplied by quantity)",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "average_price",
            "description": "Return the average price of all products",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "total_revenue",
            "description": "Return the total revenue across all products",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

# Tool dispatcher

In [0]:
def call_tool(name: str):
    if name == "max_revenue_product":
        return max_revenue_product()
    if name == "average_price":
        return average_price()
    if name == "total_revenue":
        return total_revenue()
    raise ValueError(f"Unknown tool: {name}")


# Run agent - loop

In [0]:
import json

def run_agent_chat_step(prompt: str, messages_history: list):
    # Přidáme nový dotaz od uživatele do stávající historie
    messages_history.append({"role": "user", "content": prompt})
    
    max_steps = 5
    step = 0
    
    while step < max_steps:
        step += 1
        
        response = openai_client.chat.completions.create(
            model=LLM_ENDPOINT_NAME,
            messages=messages_history, # Používáme sdílenou historii
            tools=TOOLS,
            temperature=0.0
        )
        
        msg = response.choices[0].message
        
        if not msg.tool_calls:
            # Finální odpověď modelu taky musíme uložit do historie, aby věděl, co nám řekl!
            messages_history.append(msg)
            return msg.content
            
        messages_history.append(msg)
        
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            print(f"-> Agent volá nástroj: {tool_name}")
            tool_result = call_tool(tool_name)
            
            messages_history.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(tool_result)
            })
            
    return "Agent překročil maximální počet kroků výpočtu."

# Test 1 - check 2 actions in one prompt

In [0]:
# Založení paměti (udělej jen jednou nahoře)
moje_konverzace = []

# První dotaz
odpoved_1 = run_agent_chat_step(
    "Vypočítej mi prosím rozdíl mezi celkovým obratem a průměrnou cenou.", 
    moje_konverzace
)
print("První odpověď:", odpoved_1)

-> Agent volá nástroj: total_revenue
-> Agent volá nástroj: average_price
První odpověď: Rozdíl mezi celkovým obratem a průměrnou cenou je 1300.0. Celkový obrat je 1500.0 a průměrná cena je 200.0.


[Trace(trace_id=tr-c503b676ead33c44190ebb6186c20bc7), Trace(trace_id=tr-542b9818aa442b01891d467dee972c4c), Trace(trace_id=tr-d6b07b3fb50b47ca29550c80f4d32e5a)]

# Test 2 - check history of conversation

In [0]:
# Druhý dotaz - navazujeme na předchozí výsledek
odpoved_2 = run_agent_chat_step(
    "A teď k tomu přičti 500.", 
    moje_konverzace
)
print("Druhá odpověď:", odpoved_2)

Druhá odpověď: Rozdíl mezi celkovým obratem a průměrnou cenou plus 500 je 1800.0.


Trace(trace_id=tr-0a301c29e9f0f2e30c3620761acae1f2)

### Check table

In [0]:
# 1. Načtení čistých dat přímo z Delta tabulky
df_delta = spark.table(f"{UC_PREFIX}.delta_table")

# 2. Okamžité zobrazení obsahu tabulky v Databricks
display(df_delta)

product,price,quantity
A,100,1
B,200,4
C,300,2
